In [1]:
!pip install unsloth

**Format the Data & Load the AI**

In [6]:
from datasets import load_dataset
from unsloth import FastLanguageModel
import torch
import json

# 1. Re-download the dataset (Fast because it's cached)
print("Loading dataset...")
dataset = load_dataset("datasetmaster/resumes", split="train")

# 2. Load the small Llama-3 model
print("Loading Llama-3 model...")
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Add LoRA adapters (Makes fine-tuning fast!)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. Format the Resume Dataset correctly
print("Formatting data for the AI...")
prompt_template = """Extract the key information from the following resume and output it as clean JSON.
### Resume Text:
{}

### Extracted JSON:
{}"""

def formatting_prompts_func(examples):
    texts = []
    # Get the number of resumes in this batch
    batch_size = len(examples["personal_info"])

    for i in range(batch_size):
        # We build a 'fake' raw text resume by gluing the dataset fields together
        raw_text = f"PERSONAL INFO:\n{examples['personal_info'][i]}\n\n"
        raw_text += f"EXPERIENCE:\n{examples['experience'][i]}\n\n"
        raw_text += f"EDUCATION:\n{examples['education'][i]}\n\n"
        raw_text += f"SKILLS:\n{examples['skills'][i]}"

        # We build the perfect JSON output for the AI to learn
        target_json = json.dumps({
            "personal_info": examples["personal_info"][i],
            "experience": examples["experience"][i],
            "education": examples["education"][i],
            "skills": examples["skills"][i]
        }, indent=2)

        # Combine them into the final prompt with the End-Of-String token
        text = prompt_template.format(raw_text, target_json) + tokenizer.eos_token
        texts.append(text)

    return { "text" : texts }

# Apply the formatting to our dataset
formatted_dataset = dataset.map(formatting_prompts_func, batched = True)

print("Model loaded and dataset successfully formatted! Ready to train.")

Loading dataset...
Loading Llama-3 model...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Formatting data for the AI...


Map:   0%|          | 0/4817 [00:00<?, ? examples/s]

Model loaded and dataset successfully formatted! Ready to train.


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Keeps our formatting strict
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 60 steps for a fast test run!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Stops it from asking for tracking accounts
    ),
)

print("Starting the training process...")
trainer_stats = trainer.train()
print("Training Complete!")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4817 [00:00<?, ? examples/s]

Starting the training process...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,817 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,1.169690
20,0.675563
30,0.466394
40,0.451175
50,0.418103
60,0.450441


Training Complete!


In [8]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# 1. A messy, fake resume we just made up
fake_resume = """
JOHN WICK
Email: jwick@continental.com
Phone: 555-0199

EXPERIENCE
Freelance "Consultant" - The Continental (2010 - Present)
- Resolved complex interpersonal conflicts globally.
- Specialized in high-stakes negotiations and risk management.

EDUCATION
Bachelor of Arts in Russian Ballet - Tarkovsky Theater (2005)

SKILLS
Marksmanship, Close Quarters Combat, Tactical Driving, Dog Training, Python.
"""

# 2. Format the prompt (leaving the JSON part empty for the AI to fill in!)
inputs = tokenizer(
[
    prompt_template.format(fake_resume, "")
], return_tensors = "pt").to("cuda")

# 3. Generate the answer
print("AI is reading the resume and extracting data...\n")
outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)

# 4. Print the final result
print("--- AI OUTPUT ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

AI is reading the resume and extracting data...



--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

--- AI OUTPUT ---
Extract the key information from the following resume and output it as clean JSON.
### Resume Text:

JOHN WICK
Email: jwick@continental.com
Phone: 555-0199

EXPERIENCE
Freelance "Consultant" - The Continental (2010 - Present)
- Resolved complex interpersonal conflicts globally.
- Specialized in high-stakes negotiations and risk management.

EDUCATION
Bachelor of Arts in Russian Ballet - Tarkovsky Theater (2005)

SKILLS
Marksmanship, Close Quarters Combat, Tactical Driving, Dog Training, Python.


### Extracted JSON:
{
  "name": "John Wick",
  "email": "jwick@continental.com",
  "phone": "555-0199",
  "location": {
    "city": "New York",
    "country": "United States"
  },
  "summary": "Highly skilled 'Consultant' with extensive experience in conflict resolution and risk management. Proficient in Python and dog training.",
  "experience": [
    {
      "company": "The Continental",
      "company_info": "Freelance",
      "title": "Freelance Consultant",
      "level"

In [9]:
# This converts your fine-tuned model into a laptop-friendly GGUF file
print("Saving model to GGUF format... (This might take 5-10 minutes)")

model.save_pretrained_gguf(
    "resume_parser_laptop_model",
    tokenizer,
    quantization_method = "q4_k_m" # q4 compresses it so it fits perfectly on laptop RAM
)

print("Saved successfully!")

Saving model to GGUF format... (This might take 5-10 minutes)
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:10<00:00, 65.42s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:51<00:00, 85.86s/it]


Unsloth: Merge process complete. Saved to `/content/resume_parser_laptop_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['resume_parser_laptop_model_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['resume_parser_laptop_model_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model resume_parser_laptop_model_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to resume_parser_laptop_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f resume_parser_laptop_model_gguf/Modelfile
Saved successfully!


In [ ]:
# Replace 'your_username' with your actual Hugging Face username
# Replace 'YOUR_TOKEN_HERE' with the Write token you just copied

model.push_to_hub_gguf(
    "MalinZZZRayWed/Llama-3-Resume-Parser-GGUF",
    tokenizer,
    quantization_method = "q4_k_m",
    token = "YOUR_HUGGING_FACE_TOKEN_HERE"
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...2-3b-instruct.Q4_K_M.gguf:   0%|          | 2.09MB / 2.02GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/MalinZZZRayWed/Llama-3-Resume-Parser-GGUF
Unsloth: Cleaning up temporary files...


'MalinZZZRayWed/Llama-3-Resume-Parser-GGUF'